# Data Preparation & Validation

This notebook loads and validates the human and model datasets, builds master
dataframes, and runs quality checks for downstream analysis.

**Run this once at the start** before running other analysis notebooks.

**Outputs:**
- Validated human_master and model_master dataframes
- Summary statistics and quality checks


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd
import numpy as np
from src.data_loaders import load_human_master, load_model_master
from src.config import get_paths

paths = get_paths()
print(f"✓ Data directory: {paths.data_dir}")
print(f"✓ Outputs directory: {paths.outputs_dir}")


✓ Data directory: C:\Users\AdamR\OneDrive\UCSB\VIU\HonorsThesis\data
✓ Outputs directory: C:\Users\AdamR\Projects\Flexible-Wisdom\outputs


## 1. Load Human Data

In [2]:
human_master = load_human_master()
print(f"✓ Loaded human data: {human_master.shape[0]} trials, {human_master['participantID'].nunique()} participants")
print(f"  Conditions: {sorted(human_master['condition'].unique())}")
print(f"  Columns: {list(human_master.columns)}")
human_master.head()

✓ Loaded human data: 36000 trials, 12 participants
  Conditions: ['100_0', '50_50', '80_20']
  Columns: ['stimID', 'condition', 'response', 'side_selected', 'cue_points', 'line1_angle', 'line2_angle', 'valid_cue', 'TP', 'participantID', 'decision']


,stimID,condition,response,side_selected,cue_points,line1_angle,line2_angle,valid_cue,TP,participantID,decision
0,100,50_50,6,1,2,14.314827,1.921956,False,True,SA,1
1,845,50_50,5,1,2,15.054317,4.222230,False,True,SA,1
2,245,50_50,4,1,1,14.314827,6.508956,True,True,SA,1
3,72,50_50,4,2,2,8.775056,15.054317,True,True,SA,1
4,469,50_50,4,2,2,4.222230,19.885165,True,True,SA,1


### Human Data Summary Statistics

In [3]:
print("Trials per participant:")
print(human_master.groupby('participantID').size().sort_values(ascending=False))
print("\nTrial outcome distribution (TP):")
print(human_master['TP'].value_counts().sort_index())
print("\nDecision distribution:")
print(human_master['decision'].value_counts().sort_index())

Trials per participant:
participantID
AG    3000
AW    3000
AZ    3000
BC    3000
CY    3000
GS    3000
HG    3000
JH    3000
KM    3000
KZ    3000
SA    3000
UR    3000
dtype: int64

Trial outcome distribution (TP):
TP
False    18000
True     18000
Name: count, dtype: int64

Decision distribution:
decision
0    16243
1    19757
Name: count, dtype: int64


## 2. Load Model Data

In [4]:
model_master = load_model_master()
print(f"✓ Loaded model data: {model_master.shape[0]} trials, {model_master['participantID'].nunique()} models")
print(f"  Models: {sorted(model_master['participantID'].unique())}")
print(f"  Columns: {list(model_master.columns)}")
model_master.head()

✓ Loaded model data: 36000 trials, 12 models
  Models: ['claude-3-5-haiku-20241022', 'claude-3-7-sonnet-20250219', 'claude-opus-4-20250514', 'claude-sonnet-4-20250514', 'gemini-2.5-flash', 'gemini-2.5-pro-angle', 'gemini-2.5-pro-decision', 'gpt-4.1-2025-04-14', 'gpt-5-2025-08-07', 'gpt-5-mini-2025-08-07', 'o3-2025-04-16', 'o4-mini-2025-04-16']
  Columns: ['stimID', 'condition', 'side_selected', 'cue_points', 'line1_angle', 'line2_angle', 'valid_cue', 'TP', 'response', 'participantID', 'decision']


,stimID,condition,side_selected,cue_points,line1_angle,line2_angle,valid_cue,TP,response,participantID,decision
0,100,50_50,1,2,14.314827,1.921956,False,True,present,claude-3-5-haiku-20241022,1
1,845,50_50,1,2,15.054317,4.222230,False,True,absent,claude-3-5-haiku-20241022,0
2,245,50_50,1,1,14.314827,6.508956,True,True,present,claude-3-5-haiku-20241022,1
3,72,50_50,2,2,8.775056,15.054317,True,True,absent,claude-3-5-haiku-20241022,0
4,469,50_50,2,2,4.222230,19.885165,True,True,absent,claude-3-5-haiku-20241022,0


### Model Data Summary Statistics

In [5]:
print("Trials per model:")
print(model_master.groupby('participantID').size().sort_values(ascending=False))
print("\nDecision distribution across models:")
print(model_master.groupby('participantID')['decision'].value_counts().sort_index())

Trials per model:
participantID
claude-3-5-haiku-20241022     3000
claude-3-7-sonnet-20250219    3000
claude-opus-4-20250514        3000
claude-sonnet-4-20250514      3000
gemini-2.5-flash              3000
gemini-2.5-pro-angle          3000
gemini-2.5-pro-decision       3000
gpt-4.1-2025-04-14            3000
gpt-5-2025-08-07              3000
gpt-5-mini-2025-08-07         3000
o3-2025-04-16                 3000
o4-mini-2025-04-16            3000
dtype: int64

Decision distribution across models:
participantID               decision
claude-3-5-haiku-20241022   0           1864
                            1           1136
claude-3-7-sonnet-20250219  0           1603
                            1           1397
claude-opus-4-20250514      0           1871
                            1           1129
claude-sonnet-4-20250514    0           1388
                            1           1612
gemini-2.5-flash            0           1463
                            1           1537
gemini-2.5

## 4. Cross-Domain Consistency Checks

In [6]:
# Verify human and model have same stimIDs and conditions
human_stims = set(human_master['stimID'].unique())
model_stims = set(model_master['stimID'].unique())

print(f"✓ Human stimIDs: {len(human_stims)}")
print(f"✓ Model stimIDs: {len(model_stims)}")
print(f"✓ StimIDs match: {human_stims == model_stims}")

# Check condition alignment
print(f"\n✓ Conditions:")
print(f"  Human: {sorted(human_master['condition'].unique())}")
print(f"  Model: {sorted(model_master['condition'].unique())}")


✓ Human stimIDs: 1000
✓ Model stimIDs: 1000
✓ StimIDs match: True

✓ Conditions:
  Human: ['100_0', '50_50', '80_20']
  Model: ['100_0', '50_50', '80_20']


## 5. Export Clean Data (Optional)
Save processed dataframes for reference or archival.

In [7]:
# Uncomment to export clean data
# human_master.to_csv(paths.outputs_dir / "human_master.csv", index=False)
# model_master.to_csv(paths.outputs_dir / "model_master.csv", index=False)
# bio_master.to_csv(paths.outputs_dir / "bio_master.csv", index=False)
# print("✓ Exported clean dataframes to outputs/")

print("✓ Data preparation complete. Ready for analysis.")

✓ Data preparation complete. Ready for analysis.
